In [25]:
# Copyright (c) 2015-present, Facebook, Inc.
# All rights reserved.
#
# This source code is licensed under the CC-by-NC license found in the
# LICENSE file in the root directory of this source tree.

import torch
import numpy as np
from datasets import load_dataset #,list_datasets, list_metrics, load_metric
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import argparse
import os

from src.utils import bool_flag
import os

from src.utils import target_offset

def load_data(dataset_name, data_folder):
    if dataset_name == "dbpedia14":
        dataset = load_dataset("csv", column_names=["label", "title", "sentence"],
                                data_files={"train": os.path.join(data_folder, "dbpedia_csv/train.csv"),
                                            "validation": os.path.join(data_folder, "dbpedia_csv/test.csv")})
        dataset = dataset.map(target_offset, batched=True)
        num_labels = 14
    elif dataset_name == "ag_news":
        dataset = load_dataset("ag_news")
        num_labels = 4
    elif dataset_name == "imdb":
        dataset = load_dataset("stanfordnlp/imdb")
        num_labels = 2
    elif dataset_name == "yelp":
        dataset = load_dataset("yelp_polarity")
        num_labels = 2
    elif dataset_name == "mnli":
        dataset = load_dataset("nyu-mll/glue")
        num_labels = 3
    dataset = dataset.shuffle(seed=0)

    return dataset, num_labels

# function for computing accuracy
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if type(predictions) == tuple:
        predictions = predictions[0]
    predictions = np.argmax(predictions, axis=1)
    acc = np.mean(predictions == labels)
    return {
        'accuracy': acc
    }


In [36]:

# Model
model_name = "google-bert/bert-base-uncased"
assert model_name in ["google-bert/bert-base-uncased", "gpt2"] # which model to use
checkpoint_folder = "checkpoint/" # folder in which to store temporary model checkpoints
result_folder = "result/" # folder in which to store trained models
tqdm = True # Use tqdm in output

# Data
data_folder = "dbpedia14" # folder in which to store data
dataset_name = "imdb" # classification dataset to use"
assert dataset_name in ["dbpedia14", "ag_news", "imdb", "yelp", "mnli"]

# Hyperparameters
lr = 2e-5 # learning rate
batch_size = 16 # batch size for training and eval
epochs =  5 # number of epochs to train for
weight_decay = 0.01 # weight decay
finetune = True # finetune the transformer; if False, only train linear layer

dataset, num_labels = load_data(dataset_name, data_folder)

In [37]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased",
                                          use_fast=True,
                                          device_map = "cuda")
model = AutoModelForSequenceClassification.from_pretrained(model_name,
                                                           num_labels=num_labels,
                                                           device_map="cuda")
if model_name == 'gpt2':
    tokenizer.padding_side = "right"
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id
if dataset_name == "mnli":
    # only evaluate on matched validation set
    testset_key = "validation_matched"
    preprocess_function = lambda examples: tokenizer(
        examples["premise"], examples["hypothesis"], max_length=256, truncation=True)
else:
    text_key = 'text' if (dataset_name in ["ag_news", "imdb", "yelp"]) else 'sentence'
    testset_key = 'test' if (dataset_name in ["ag_news", "imdb", "yelp"]) else 'validation'
    preprocess_function = lambda examples: tokenizer(examples[text_key], max_length=256, truncation=True)
encoded_dataset = dataset.map(preprocess_function, batched=True)

train_args = TrainingArguments(
    checkpoint_folder,
    disable_tqdm=not tqdm,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    weight_decay=weight_decay,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

trainer = Trainer(
    model,
    train_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset[testset_key],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

if not finetune:
    # freeze parameters of transformer
    transformer = list(model.children())[0]
    for param in transformer.parameters():
        param.requires_grad = False

trainer.train()
trainer.evaluate()
suffix = ''
if finetune:
    suffix += '_finetune'
torch.save(model.state_dict(),
            os.path.join(result_folder, "%s_%s%s.pth" % (model_name.replace('/', '-'), dataset, suffix)))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
dataset_name in ["mnli","imdb"]
dataset_name

In [30]:
torch.cuda.is_available()

True